In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS fraud_silver")

DataFrame[]

In [0]:
transactions = spark.table("fraud_bronze.bronze_transactions")

identity = spark.table("fraud_bronze.bronze_identity")

customer = spark.table("fraud_bronze.bronze_customer_master")

merchant = spark.table("fraud_bronze.bronze_merchant_master")

country = spark.table("fraud_bronze.bronze_country_risk")

exchange = spark.table("fraud_bronze.bronze_exchange_rates")

ofac = spark.table("fraud_bronze.bronze_ofac")

pipeline = spark.table("fraud_bronze.bronze_pipeline_config")

In [0]:
from pyspark.sql.functions import *

silver_customer = (
    customer
    .dropDuplicates(["customer_id"])
    .withColumn("created_ts", current_timestamp())
)

In [0]:
silver_customer.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("fraud_silver.silver_customer_master")

In [0]:
silver_merchant = (
    merchant
    .dropDuplicates(["merchant_id"])
    .withColumn("created_ts", current_timestamp())
)

In [0]:
silver_merchant.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("fraud_silver.silver_merchant_master")

In [0]:
silver_country = (
    country
    .dropDuplicates(["country_code"])
    .withColumn("created_ts", current_timestamp())
)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5381395903405162>, line 2
      1 silver_country = (
----> 2     country
      3     .dropDuplicates(["country_code"])
      4     .withColumn("created_ts", current_timestamp())
      5 )

NameError: name 'country' is not defined

In [0]:
silver_country.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("fraud_silver.silver_country_risk")

In [0]:
silver_exchange = (
    exchange
    .dropDuplicates(["currency_code"])
    .withColumn("created_ts", current_timestamp())
)

In [0]:
silver_exchange.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("fraud_silver.silver_exchange_rates")

In [0]:
silver_ofac = (
    ofac
    .dropDuplicates(["watchlist_id"])
    .withColumn("created_ts", current_timestamp())
)

In [0]:
silver_ofac.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("fraud_silver.silver_ofac")

In [0]:
silver_pipeline = (
    pipeline
    .dropDuplicates(["pipeline_name"])
    .withColumn("created_ts", current_timestamp())
)

In [0]:
silver_pipeline.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("fraud_silver.silver_pipeline_config")

In [0]:
tables = [

"silver_customer_master",
"silver_merchant_master",
"silver_country_risk",
"silver_exchange_rates",
"silver_ofac",
"silver_pipeline_config"

]

for table in tables:

    cnt = spark.sql(f"""
    SELECT COUNT(*) cnt
    FROM fraud_silver.{table}
    """).collect()[0]["cnt"]

    print(f"{table:35} {cnt}")

silver_customer_master              14893
silver_merchant_master              269
silver_country_risk                 30
silver_exchange_rates               15
silver_ofac                         15
silver_pipeline_config              6
